## 統計製程管制 (SPC) 圖

- 目標：掌握統計製程管制（SPC）的核心理論，利用 Python **自動繪製 \(\bar{X}-R\)（平均值-全距）管制圖**，並精準**計算製程能力指數 \(C*{pk}\) 與 \(P*{pk}\)，以量化評估測試機台的穩定度與產能品質**。


### 1. X-bar 圖與 R 圖自動化計算與繪製

- 概念：
    - 子組邏輯：不是對單一數值做對照，而是每次抽樣一組（如 n=5 顆晶粒），計算組內平均（X-bar）與組內全距（R = 組內最大值－最小值），然後監控製程中心是否偏移與製程變異是否變大。
    - 控制界限計算：X-bar 圖與 R 圖的不同控制界限（UCL/LCL）是用「平均全距 R-bar」乘上對應樣本數 n 查表得到的控制係數（A2、D3、D4）計算出，不是用數據本身的標準差直接計算3-sigma。
    - 判讀規則：不只是「超出才算異常」，還有連續多點同側依序（界限）、連續上升或下降趨勢等額外規則，用於提早探測製程變異，而非等到真正超規才做出反應。
- 實作：頻寬測試整合工程師需要監控高頻晶圓測試機台的電性輸出。我們每隔一小時抽樣 5 顆晶粒（樣本數 \(n=5\)）作為一個子組（Subgroup），連續收集 10 個組別。當**製程出現異常漂移**時，SPC 管制圖必須能**自動觸發警報**。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 模擬 10 組 NPI 階段的頻寬測試數據（每組包含 5 個樣本，單位：GHz）
# 故意讓第 8 組數據產生向上的製程漂移
np.random.seed(42)
data = np.random.normal(loc=28.0, scale=0.5, size=(10, 5))
data[7] += 1.2  # 模擬第 8 組機台突發性異常

# 建立 DataFrame
subgroup_ids = [f"Subgroup_{i + 1}" for i in range(10)]
df_spc = pd.DataFrame(
    data, index=subgroup_ids, columns=[f"Sample_{i + 1}" for i in range(5)]
)

# 計算每組的平均值 (X-bar) 與 全距 (R)
df_spc["X_bar"] = df_spc.mean(axis=1)
df_spc["R"] = df_spc.max(axis=1) - df_spc.min(axis=1)

# 計算總平均值 (X-double-bar) 與 平均全距 (R-bar)
X_double_bar = df_spc["X_bar"].mean()
R_bar = df_spc["R"].mean()

# 根據統計學常數表，當 n=5 時：A2 = 0.577, D3 = 0, D4 = 2.114
A2, D3, D4 = 0.577, 0.0, 2.114

# 計算 X-bar 圖的管制界限
X_UCL = X_double_bar + A2 * R_bar
X_LCL = X_double_bar - A2 * R_bar

# 計算 R 图的管制界限
R_UCL = D4 * R_bar
R_LCL = D3 * R_bar

print(f"統計參數：X_double_bar = {X_double_bar:.3f}, R_bar = {R_bar:.3f}")
print(
    f"X-bar 界限：UCL = {X_UCL:.3f}, LCL = {X_LCL:.3f}"
)

# 繪製 X-bar 管制圖
plt.figure(figsize=(10, 4))
plt.plot(df_spc["X_bar"], marker="o", color="b", label="子組平均值 ($\bar{X}$)")
plt.axhline(y=X_double_bar, color="green", linestyle="-", label="中心線 (CL)")
plt.axhline(y=X_UCL, color="red", linestyle="--", label="上管制界限 (UCL)")
plt.axhline(y=X_LCL, color="red", linestyle="--", label="下管制界限 (LCL)")

# 自動標註超出管制界限的異常點（WECO 規則 1：超出 3 個標準差）
for idx, row in df_spc.iterrows():
    if row["X_bar"] > X_UCL or row["X_bar"] < X_LCL:
        plt.plot(idx, row["X_bar"], marker="s", color="orange", markersize=10)
        plt.text(idx, row["X_bar"] + 0.1, "OOC 異常", color="darkred", weight="bold")

plt.title("𠹹自動生成：$\bar{X}$ 平均值管制圖 (X-bar Chart)", fontsize=12)
plt.ylabel("頻寬 (GHz)")
plt.grid(True, alpha=0.3)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 2. 製程能力指數 Cpk 與 Ppk 精準計算

- 概念：
    - Cpk（短期製程能力）：使用組內變異（由平均全距 bar{R}/d_2 估算群體標準差 σ\*{within}），反映機台在短期內排除組間干擾下的純粹潛在能力。
    - Ppk（長期製程效能）：使用整體數據的樣本標準差（σ\*{total}，由 np.std(ddof=1) 計算），反映包含機台環境漂移、材料波動後的實際全盤表現。
    - 與規格界限的關係：Cpk/Ppk 均取製程分配跟規格上下限（USL/LSL）比較，數值越大代表製程能力越強（越難超出規格）；使用中常見的比率是 Cpk≥1.33 視為合格。
    - 公式重點：取「製程中心到較近規格界限的距離」除以「3倍的標準差」，取兩邊（靠近USL與靠近LSL）中較小的一個，代表最壞情況下的能力。
    - Cpk 與 Ppk 差異的實際意義：若 Cpk 遠大於 Ppk，代表機台短期很穩定，但長期有其（換班、換料、環境溫潮變化造成的組間變異），這種落差本身就是工程調查的線索。


In [ ]:
# 定義產品的規格界限 (Specification Limits)
USL = 30.0  # 規格上限 (Upper Specification Limit)
LSL = 26.0  # 規格下限 (Lower Specification Limit)

# 將所有原始點位攤平，用以計算整體統計量
all_samples = data.flatten()
overall_mean = np.mean(all_samples)

# 短期標準差估算 (組內變異)：當 n=5 時，d2 = 2.326
d2 = 2.326
sigma_within = R_bar / d2

# 長期標準差計算 (總體變異)
sigma_total = np.std(all_samples, ddof=1)

# 計算 Cpk
Cpu_short = (USL - overall_mean) / (3 * sigma_within)
Cpl_short = (overall_mean - LSL) / (3 * sigma_within)
Cpk = min(Cpu_short, Cpl_short)

# 計算 Ppk
Ppu_long = (USL - overall_mean) / (3 * sigma_total)
Ppl_long = (overall_mean - LSL) / (3 * sigma_total)
Ppk = min(Ppu_long, Ppl_long)

print("=" * 40)
print(f"產品設計規格：LSL={LSL} GHz ~ USL={USL} GHz")
print(f"實際製程平均值：{overall_mean:.3f} GHz")
print(f"短期變異 (Sigma Within): {sigma_within:.4f} -> 估算 Cpk = {Cpk:.3f}")
print(f"長期變異 (Sigma Total) : {sigma_total:.4f} -> 計算 Ppk = {Ppk:.3f}")
print("=" * 40)

# 品質水準判定
if Cpk >= 1.33:
    print("判定結果：製程能力充足 (Cpk >= 1.33)，機台狀況良好。")
else:
    print("判定結果：製程能力不足 (Cpk < 1.33)，必須重新校正測試探針床！")

- 總結：因為我有材料實驗室與硬體儀器的實務背景，我很清楚手動判讀製程圖表的侷限。因此在我的專案中，我用 Python 實作了 SPC 自動化管制模組。我利用統計學常數（如 \(A*2, D_4\)），讓程式能動態計算出 \(\bar{X}-R\) 管制圖的 UCL 與 LCL，並能即時揪出超出 3 個標準差的 Out-of-Control (OOC) 點位。更重要的是，我能區分 \(C*{pk}\) 與 \(P*{pk}\) 的底層物理意義差異：\(C*{pk}\) 透過組內全距估算短期的潛在機台能力，而 \(P\_{pk}\) 採用全體樣本標準差來評估長期包含環境漂移後的真實良率表现。這種將統計理論程式化的能力，能大幅縮短新產品引進（NPI）階段的製程除錯時間。
